# 01 · Data collection
### Chess upsets over the board (DA2402 course project)

This notebook collects the two data sources and turns every downloaded file into a table without changing any value. It is the first of four notebooks:

| Notebook | Does |
|---|---|
| 01_collection (this one) | downloads the Lichess broadcast PGN files and the FIDE rating lists, records robots.txt / terms, writes the manifest, parses the raw files into Parquet, combines them, and profiles every raw table |
| 02_cleaning | cleaning rules, missing values, duplicates, outliers, cleaning log, final clean table and data dictionary |
| 03_model | upset model and critical gaps |
| 04_report | results and figures |

**Run:** *Kernel > Restart & Run All*. Every step is resumable: a file that is already downloaded or already parsed is skipped, so the notebook can be stopped and re-run at any time. A full run from an empty `data/` folder downloads about 2.5 GB (about 324 files; the time depends on the connection) and then needs about 15-25 minutes to parse.

**Outputs**

| Folder | Files |
|---|---|
| `data/raw/` | `lichess/lichess_db_broadcast_YYYY-MM.pgn.zst`, `fide/<list>_<mon><yy>frl_xml.zip`, `compliance/` (robots.txt and terms pages), `MANIFEST.csv` |
| `data/interim/` | `games_YYYY-MM.parquet`, `moves_YYYY-MM.parquet`, `fide_<list>_YYYY-MM.parquet`, `all_games.parquet`, `fide_ratings.parquet`, `fide_players.parquet` |
| `reports/` | `raw_profile.csv` (shape, dtype, missing count and meaning of every raw column), `games_per_month.csv` |

## 0. Configuration

The only cell to edit. `DATA_ROOT` points to the folder that holds `raw/`, `interim/` and `clean/`; it can also be set from outside with the environment variable `CHESS_DATA_ROOT` (used for test runs on a copy of the data). `MONTHS` selects the months to collect: by default all 81 months 2020-01 to 2026-09; a short list (for example `CHESS_MONTHS=2020-01,2023-01`) gives a quick test run.

In [1]:
import os

DATA_ROOT = os.environ.get("CHESS_DATA_ROOT", os.path.join("..", "data"))   # holds raw/ interim/ clean/
REPORTS   = os.environ.get("CHESS_REPORTS", os.path.join("..", "reports"))   # small text/CSV reports
RAW, INTERIM, CLEAN = (os.path.join(DATA_ROOT, d) for d in ("raw", "interim", "clean"))

FIRST_MONTH, LAST_MONTH = "2020-01", "2026-09"                 # study period (81 months)
MONTHS_ENV = os.environ.get("CHESS_MONTHS", "")                 # e.g. "2020-01,2023-01" for a test run
DOWNLOAD = os.environ.get("CHESS_DOWNLOAD", "1") == "1"         # False: never use the network, work with the files in raw/
FIDE_LISTS = ("standard", "rapid", "blitz")

In [2]:
import re, io, glob, time, zipfile, hashlib, datetime, platform
import xml.etree.ElementTree as ET
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.compute as pc, pyarrow.parquet as pq
from IPython.display import display, Markdown

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.max_rows", 80)
def say(text): display(Markdown(text))          # prose whose numbers are computed from the data

def month_range(a, b):
    y, m = map(int, a.split("-")); y2, m2 = map(int, b.split("-"))
    out = []
    while (y, m) <= (y2, m2):
        out.append(f"{y}-{m:02d}"); y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out
MONTHS = [m.strip() for m in MONTHS_ENV.split(",") if m.strip()] or month_range(FIRST_MONTH, LAST_MONTH)
for d in (RAW, INTERIM, CLEAN, REPORTS, os.path.join(RAW, "lichess"), os.path.join(RAW, "fide"), os.path.join(RAW, "compliance")):
    os.makedirs(d, exist_ok=True)
print(f"python {platform.python_version()} | pandas {pd.__version__} | pyarrow {pa.__version__} | run on {datetime.date.today()}")
print(f"DATA_ROOT = {os.path.abspath(DATA_ROOT)} | {len(MONTHS)} months: {MONTHS[0]} .. {MONTHS[-1]} | downloads {'on' if DOWNLOAD else 'off'}")

python 3.13.5 | pandas 3.1.0 | pyarrow 26.0.0 | run on 2026-11-05
DATA_ROOT = C:\dev\chess-upsets\submission\data | 81 months: 2020-01 .. 2026-09 | downloads on


## 1. Sources

| | Lichess broadcast database | FIDE rating lists |
|---|---|---|
| URL | https://database.lichess.org/#broadcasts (files `https://database.lichess.org/broadcast/lichess_db_broadcast_YYYY-MM.pgn.zst`) | https://ratings.fide.com/download_lists.phtml (files `https://ratings.fide.com/download/<list>_<mon><yy>frl_xml.zip`) |
| Content | One PGN file per month with the games of over-the-board events relayed on Lichess (tags, moves, clock and engine comments) | One list per month and time control (standard, rapid, blitz) of all rated players: rating, games, K-factor, title, federation, birth year |
| Format | Zstandard-compressed PGN (`.pgn.zst`) | `.zip` holding one XML file |
| Period | 2020-01 to 2026-09 (81 files) | the same 81 months, three lists each (243 files) |
| Licence | CC BY-SA 4.0 for the broadcast games (statement on the broadcast panel of the database page; the page header says the database exports in general are CC0) | no terms of use published; the site footer reserves all rights (quoted in section 2) |
| Collected | download date of every file is in `MANIFEST.csv` (section 4); the files used in the project were downloaded on 2026-10-24 | same, 2026-10-24 |

The Lichess games are the primary data. The FIDE lists are used to fill and check ratings; they are joined to the games on the FIDE id, which both sources carry, not on player names.

## 2. robots.txt and terms of use

The robots.txt files and the terms pages of both sources are saved into `raw/compliance/` as served, with the date in the file name. The cell fetches a page only if no saved copy exists, so a re-run does not overwrite the evidence; the evidence used for this project was saved on 2026-10-31.

In [3]:
PAGES = {"lichess_robots":          "https://lichess.org/robots.txt",
         "lichess_database_robots": "https://database.lichess.org/robots.txt",
         "lichess_terms":           "https://lichess.org/terms-of-service",
         "lichess_database_page":   "https://database.lichess.org/",
         "fide_robots":             "https://ratings.fide.com/robots.txt",
         "fide_ratings_page":       "https://ratings.fide.com/download_lists.phtml"}
CP = os.path.join(RAW, "compliance")

def saved(name):
    """Newest saved copy of a compliance page, or None."""
    f = sorted(glob.glob(os.path.join(CP, name + "_*")))
    return f[-1] if f else None
def text(path): return open(path, encoding="utf-8", errors="replace").read()
def plain(html): return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", html))
def quote(pattern, s):
    m = re.search(pattern, s)
    return m.group(0) if m else "(passage not found in the saved file)"

In [4]:
rows = []
for name, url in PAGES.items():
    if saved(name):
        rows.append((name, url, os.path.basename(saved(name)), "saved copy exists, not fetched again")); continue
    if not DOWNLOAD:
        rows.append((name, url, None, "missing (DOWNLOAD is off)")); continue
    try:
        import requests
        r = requests.get(url, timeout=30, headers={"User-Agent": "DA2402 student project"})
        fn = f"{name}_{datetime.date.today()}.{'txt' if 'robots' in name else 'html'}"
        open(os.path.join(CP, fn), "wb").write(r.content)
        rows.append((name, url, fn, f"fetched now, HTTP {r.status_code}"))
    except Exception as e:
        rows.append((name, url, None, f"FAILED ({type(e).__name__}); save the page by hand"))
display(pd.DataFrame(rows, columns=["page", "url", "saved file", "status"]))

,page,url,saved file,status
0,lichess_robots,https://lichess.org/robots.txt,lichess_robots_2026-10-31.txt,"saved copy exists, not fetched again"
1,lichess_database_robots,https://database.lichess.org/robots.txt,lichess_database_robots_2026-10-31.txt,"saved copy exists, not fetched again"
2,lichess_terms,https://lichess.org/terms-of-service,lichess_terms_2026-10-31.html,"saved copy exists, not fetched again"
3,lichess_database_page,https://database.lichess.org/,lichess_database_page_2026-10-31.html,"saved copy exists, not fetched again"
4,fide_robots,https://ratings.fide.com/robots.txt,fide_robots_2026-10-31.txt,"saved copy exists, not fetched again"
5,fide_ratings_page,https://ratings.fide.com/download_lists.phtml,fide_ratings_page_2026-10-31.html,"saved copy exists, not fetched again"


The passages that matter, quoted from the saved files:

In [5]:
print("== https://lichess.org/robots.txt  (" + os.path.basename(saved("lichess_robots")) + ")")
print(text(saved("lichess_robots")).strip())
print("\n== https://database.lichess.org/robots.txt: server reply ->", plain(text(saved("lichess_database_robots"))).strip()[:60])

db = plain(text(saved("lichess_database_page")))
print("\n== https://database.lichess.org/ (page header):")
print("  ", quote(r"Database exports are released under the.*?without asking for permission\.", db))
print("== https://database.lichess.org/#broadcasts (broadcast panel):")
print("  ", quote(r"[\d,]+ games in PGN format, from the official Lichess broadcasts.*?license \.?", db))

tos = plain(text(saved("lichess_terms")))
print("\n== https://lichess.org/terms-of-service:")
print("  ", quote(r"Last Modified:.*?Enjoy the site!", tos))

fide = plain(text(saved("fide_ratings_page")))
print("\n== https://ratings.fide.com/download_lists.phtml, footer:")
print("  ", quote(r"© \d{4} FIDE International Chess Federation\. All Rights Reserved\..*?written permission of FIDE[^.]*\.", fide))
print("\n== https://ratings.fide.com/robots.txt: server reply ->", quote(r"4xx The page you are looking for does not exist\.", plain(text(saved("fide_robots")))))

== https://lichess.org/robots.txt  (lichess_robots_2026-10-31.txt)
User-agent: *
Allow: /
Disallow: /game/
Disallow: /@/*/download
Disallow: /games/export/
Disallow: /api/
Disallow: /opening/config/
Disallow: /study/search
Disallow: /study/embed/
Disallow: /embed/
Disallow: /video?*
Disallow: /training/of-player
Allow: /game/export/gif/thumbnail/

== https://database.lichess.org/robots.txt: server reply -> 404 Not Found 404 Not Found nginx

== https://database.lichess.org/ (page header):
   Database exports are released under the Creative Commons CC0 license . Use them for research, commercial purpose, publication, anything you like. You can download, modify and redistribute them, without asking for permission.
== https://database.lichess.org/#broadcasts (broadcast panel):
   1,263,913 games in PGN format, from the official Lichess broadcasts . Broadcast games are released under the Creative Commons Attribution-ShareAlike 4.0 license .

== https://lichess.org/terms-of-service:
   Last 

**Reading.**
- *Lichess.* `lichess.org/robots.txt` disallows `/api/`, `/game/`, `/games/export/`, the per-player download and a few study and embed paths; none of them is used. The monthly files come from the separate host `database.lichess.org`, which publishes no robots.txt (HTTP 404); it is a file-download service, and the files are offered for direct download. The page header says database exports are released under CC0, but the broadcast panel states that broadcast games are released under CC BY-SA 4.0. The stricter, more specific statement is followed: the source is credited (README, report), and data derived from the games are shared under the same licence. The Lichess terms of service concern the use of the playing site (accounts, fair play) and contain nothing about the database exports.
- *FIDE.* `ratings.fide.com` publishes no robots.txt (the reply is an error page) and no terms of use for the rating lists; the footer reserves all rights and forbids reproduction without written permission. The lists are therefore downloaded once each, by script, with a pause of one second between requests, used only for this non-commercial coursework, and not redistributed (the data folders are not part of the repository; the team's download link points to the Lichess-derived files only, see README).

## 3. Download (resumable)

`fetch` downloads one URL into `raw/`. A finished file is skipped; a download is written to a `.part` file and renamed only when it is complete (its size equals the server's `Content-Length`), and an interrupted `.part` file is resumed with an HTTP range request. A FIDE file must start with the zip signature `PK`, otherwise it is an error page and is discarded. FIDE months that are not offered are written to `raw/fide/_missing.txt` (information for the report, not an error).

In [6]:
HEAD = {"User-Agent": "Mozilla/5.0 (university course project, DA2402)"}
MON = "jan feb mar apr may jun jul aug sep oct nov dec".split()

def lichess_url(month):                       # month = "YYYY-MM"
    return f"https://database.lichess.org/broadcast/lichess_db_broadcast_{month}.pgn.zst"

def fide_url(lst, month):                     # lst = standard | rapid | blitz
    y, m = map(int, month.split("-"))
    return f"https://ratings.fide.com/download/{lst}_{MON[m - 1]}{y % 100:02d}frl_xml.zip"

def fetch(url, path, must_be_zip=False):
    """Download url to path. Skips a finished file, resumes a .part file, renames only when complete."""
    if os.path.exists(path):
        return "skipped (already downloaded)"
    if not DOWNLOAD:
        return "missing (downloads switched off)"
    import requests
    part = path + ".part"
    have = os.path.getsize(part) if os.path.exists(part) else 0
    headers = dict(HEAD, **({"Range": f"bytes={have}-"} if have else {}))
    total = None
    with requests.get(url, headers=headers, stream=True, timeout=120) as r:
        if r.status_code == 404:
            return "404 (not offered)"
        if r.status_code != 416:              # 416 = the .part file already holds the whole file
            r.raise_for_status()
            if r.status_code == 206:          # partial content: append to the .part file
                total = r.headers.get("Content-Range", "/").split("/")[-1]
                mode = "ab"
            else:
                total = r.headers.get("Content-Length"); mode = "wb"
            with open(part, mode) as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
    size = os.path.getsize(part)
    if total and total.isdigit() and int(total) != size:
        return f"incomplete ({size:,} of {int(total):,} bytes): run the cell again to resume"
    if must_be_zip:
        with open(part, "rb") as f:
            if f.read(2) != b"PK":
                os.remove(part); return "not a zip (error page), discarded"
    os.replace(part, path)
    return f"downloaded ({size / 1e6:.1f} MB)"

In [7]:
log = []
for month in MONTHS:
    url = lichess_url(month)
    try:    status = fetch(url, os.path.join(RAW, "lichess", os.path.basename(url)))
    except Exception as e: status = f"error: {type(e).__name__}: {e}"
    log.append(("Lichess", month, url, status))
    for lst in FIDE_LISTS:
        url = fide_url(lst, month)
        try:    status = fetch(url, os.path.join(RAW, "fide", os.path.basename(url)), must_be_zip=True)
        except Exception as e: status = f"error: {type(e).__name__}: {e}"
        log.append((f"FIDE {lst}", month, url, status))
        if status.startswith(("downloaded", "404", "not a zip", "incomplete")):
            time.sleep(1)                     # be polite to the FIDE server
dl = pd.DataFrame(log, columns=["source", "month", "url", "status"])
dl["result"] = dl["status"].str.extract(r"^(\w+)")[0]
display(dl.groupby(["source", "result"]).size().unstack(fill_value=0))
problems = dl[~dl.result.isin(["skipped", "downloaded"])]
open(os.path.join(RAW, "fide", "_missing.txt"), "w").write(
    "\n".join(f"{r.source} {r.month} {r.status}" for r in problems.itertuples() if r.source.startswith("FIDE")))
if len(problems):
    display(problems.head(20))
say(f"**{(dl.result == 'downloaded').sum()}** files downloaded now, **{(dl.result == 'skipped').sum()}** already present, **{len(problems)}** not obtained.")

result,skipped
source,
FIDE blitz,81
FIDE rapid,81
FIDE standard,81
Lichess,81


**0** files downloaded now, **324** already present, **0** not obtained.

## 4. Manifest of the raw files

Every raw file is listed with its source, period, URL, size, download date and a SHA-256 checksum. The download date is the file's modification time: the files are written once by the downloader and never edited (the checksum lets anyone verify that). File names carry the source and the period.

In [8]:
def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

rows = []
for f in sorted(os.listdir(os.path.join(RAW, "lichess"))):
    m = re.fullmatch(r"lichess_db_broadcast_(\d{4}-\d{2})\.pgn\.zst", f)
    if m: rows.append(("Lichess broadcasts", "PGN games", m[1], lichess_url(m[1]), f"lichess/{f}"))
for f in sorted(os.listdir(os.path.join(RAW, "fide"))):
    m = re.fullmatch(r"(standard|rapid|blitz)_([a-z]{3})(\d{2})frl_xml\.zip", f)
    if m: rows.append(("FIDE rating lists", f"{m[1]} list (XML)", f"20{m[3]}-{MON.index(m[2]) + 1:02d}", f"https://ratings.fide.com/download/{f}", f"fide/{f}"))
man = pd.DataFrame(rows, columns=["source", "content", "period", "url", "path"])
full = man["path"].map(lambda p: os.path.join(RAW, p))
man["size_mb"] = full.map(lambda p: round(os.path.getsize(p) / 1e6, 2))
man["downloaded"] = full.map(lambda p: datetime.date.fromtimestamp(os.path.getmtime(p)).isoformat())
man["sha256"] = full.map(sha256)
man.to_csv(os.path.join(RAW, "MANIFEST.csv"), index=False)
display(man.groupby(["source", "content"]).agg(files=("path", "size"), first=("period", "min"), last=("period", "max"),
        size_mb=("size_mb", "sum"), downloaded_from=("downloaded", "min"), downloaded_to=("downloaded", "max")).round(1))
display(man.sample(min(4, len(man)), random_state=1).sort_values("period")[["source", "period", "url", "size_mb", "downloaded"]])

files    first     last  size_mb downloaded_from downloaded_to
source             content                                                                            
FIDE rating lists  blitz list (XML)        81  2020-01  2026-09    413.8      2026-10-24    2026-10-24
                   rapid list (XML)        81  2020-01  2026-09    585.6      2026-10-24    2026-10-24
                   standard list (XML)     81  2020-01  2026-09    876.4      2026-10-24    2026-10-24
Lichess broadcasts PGN games               81  2020-01  2026-09    744.2      2026-10-24    2026-10-24

,source,period,url,size_mb,downloaded
184,FIDE rating lists,2022-02,https://ratings.fide.com/download/rapid_feb22frl_xml.zip,5.37,2026-10-24
232,FIDE rating lists,2022-10,https://ratings.fide.com/download/rapid_oct22frl_xml.zip,6.00,2026-10-24
213,FIDE rating lists,2023-03,https://ratings.fide.com/download/rapid_mar23frl_xml.zip,6.46,2026-10-24
139,FIDE rating lists,2023-05,https://ratings.fide.com/download/blitz_may23frl_xml.zip,4.79,2026-10-24


## 5. First look at the raw files

Before any change, one raw file of each kind is opened. A PGN record is a block of tags followed by the move text, in which Lichess stores the players' clock (`[%clk h:mm:ss]`) and, for some games, an engine evaluation (`[%eval]`) after each move:

In [9]:
import zstandard

def open_text(path):
    """Text stream of a .pgn.zst (Zstandard) or plain .pgn file; streams, so memory stays small."""
    if path.endswith(".zst"):
        reader = zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(open(path, "rb"), read_across_frames=True)
        return io.TextIOWrapper(reader, encoding="utf-8", errors="replace")
    return open(path, encoding="utf-8", errors="replace")

In [10]:
first_pgn = os.path.join(RAW, "lichess", os.path.basename(lichess_url(MONTHS[0])))
lines = []
with open_text(first_pgn) as f:
    for line in f:
        if line.startswith("[Event ") and lines: break
        lines.append(line.rstrip("\n"))
print(f"first game of {os.path.basename(first_pgn)}:\n")
print("\n".join(l if len(l) < 160 else l[:157] + "..." for l in lines))

first game of lichess_db_broadcast_2020-01.pgn.zst:

[Event "FIDE Women's World Championship Match 2020"]
[Site "Shanghai CHN"]
[Date "2020.01.05"]
[Round "1"]
[White "Goryachkina, Aleksandra"]
[Black "Ju, Wenjun"]
[Result "1/2-1/2"]
[WhiteElo "2578"]
[WhiteTitle "GM"]
[WhiteFideId "4147103"]
[BlackElo "2584"]
[BlackTitle "GM"]
[BlackFideId "8603006"]
[TimeControl "40/5400+30:1800+30"]
[Variant "Standard"]
[ECO "D37"]
[Opening "Queen's Gambit Declined: Three Knights Variation"]
[StudyName "Women's World Chess Championship 2020 - Game 1"]
[ChapterName "Goryachkina, Aleksandra - Ju, Wenjun"]
[BroadcastName "Women's World Chess Championship 2020 - Game 1"]
[BroadcastURL "https://lichess.org/broadcast/womens-world-chess-championship-2020-game-1/-/6E03pUdi"]
[GameURL "https://lichess.org/broadcast/womens-world-chess-championship-2020-game-1/-/6E03pUdi/caQhB9QT"]

1. d4 { [%eval 0.25] [%clk 1:30:43] } 1... Nf6 { [%eval 0.22] [%clk 1:30:42] } 2. c4 { [%eval 0.18] [%clk 1:31:03] } 2... e6 { [%

A FIDE list is a zipped XML file with one `<player>` record per rated player; empty elements mean "none" (no title, no flag):

In [11]:
zpath = os.path.join(RAW, "fide", os.path.basename(fide_url("standard", MONTHS[0])))
with zipfile.ZipFile(zpath) as z:
    inner = [i for i in z.infolist() if i.filename.lower().endswith(".xml")][0]
    print(f"{os.path.basename(zpath)} holds {inner.filename} ({inner.file_size / 1e6:.0f} MB uncompressed)\n")
    with z.open(inner) as f:
        print(f.read(700).decode("utf-8", "replace"))

standard_jan20frl_xml.zip holds standard_jan20frl_xml.xml (100 MB uncompressed)

<playerslist>
<player>
<fideid>25121731</fideid>
<name>A C J John</name>
<country>IND</country>
<sex>M</sex>
<title></title>
<w_title></w_title>
<o_title></o_title>
<foa_title></foa_title>
<rating>1063</rating>
<games>0</games>
<k>40</k>
<birthday>1987</birthday>
<flag></flag>
</player>
<player>
<fideid>35077023</fideid>
<name>A Chakravarthy</name>
<country>IND</country>
<sex>M</sex>
<title></title>
<w_title></w_title>
<o_title></o_title>
<foa_title></foa_title>
<rating>1151</rating>
<games>0</games>
<k>40</k>
<birthday>1986</birthday>
<flag>i</flag>
</player>
<player>
<fideid>10207538</fideid>
<name>A E M, Doshtagir</name>
<country>BAN</country>
<sex>M</sex>
<title></title>
<w_title></w_tit


## 6. Parsing the raw files into tables

### 6.1 Lichess PGN to `games_YYYY-MM.parquet` and `moves_YYYY-MM.parquet`

Each monthly PGN file is streamed game by game. Every tag is kept as written (tag names converted to `snake_case`, e.g. `WhiteFideId` becomes `white_fide_id`); only the two rating tags are converted to integers (a non-numeric rating becomes missing). From the move text the parser *counts* (plies, `?!`/`?`/`??` annotations) and *reads* the first and last clock of each side; it applies no chess rules. The move text itself goes to a separate `moves_` table (joined on `game_url`) because it is large and only needed for the clock estimate in notebook 02.

A month is parsed again if its raw file is newer than its Parquet file (a re-downloaded file is never left with a stale table).

In [12]:
TAG  = re.compile(r'^\[(\w+) "(.*)"\]\s*$')
MOVE = re.compile(r'(\d+)(\.{1,3})\s*([^\s{]+)((?:\s*\{[^}]*\})*)')      # move number, dots, SAN, comments
CLK  = re.compile(r'\[%clk (\d+):(\d+):(\d+(?:\.\d+)?)\]')
EVAL = re.compile(r'\[%eval ([^\]]+)\]')

def snake(t):                                   # WhiteFideId -> white_fide_id, GameURL -> game_url
    t = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", t)
    return re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", t).lower()

def clk_seconds(comments):
    m = CLK.search(comments)
    return int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3)) if m else None

def analyse_moves(text):
    """Counts and clock facts from the move text."""
    out = dict(n_plies=0, has_clk=False, has_eval=False, first_clk_w=None, first_clk_b=None, last_clk_w=None, last_clk_b=None,
               n_inacc_w=0, n_mistake_w=0, n_blunder_w=0, n_inacc_b=0, n_mistake_b=0, n_blunder_b=0)
    for num, dots, san, comments in MOVE.findall(text):
        side = "w" if dots == "." else "b"
        out["n_plies"] += 1
        if san.endswith("??"):   out[f"n_blunder_{side}"] += 1
        elif san.endswith("?!"): out[f"n_inacc_{side}"] += 1
        elif san.endswith("?"):  out[f"n_mistake_{side}"] += 1
        if comments:
            if EVAL.search(comments): out["has_eval"] = True
            c = clk_seconds(comments)
            if c is not None:
                out["has_clk"] = True
                if out[f"first_clk_{side}"] is None: out[f"first_clk_{side}"] = c
                out[f"last_clk_{side}"] = c
    return out

def pgn_games(f):
    """Yield (tags, movetext) for every game of an open PGN stream; a game starts at an [Event ...] line."""
    tags, moves = {}, []
    for line in f:
        if line.startswith("[Event ") and (tags or moves):
            yield tags, " ".join(moves).strip()
            tags, moves = {}, []
        m = TAG.match(line)
        if m:   tags[m.group(1)] = m.group(2)
        elif line.strip(): moves.append(line.strip())
    if tags:
        yield tags, " ".join(moves).strip()

def write_parquet(df, path):                    # write to a temporary file first, so an interrupted run leaves no broken file
    df.to_parquet(path + ".tmp", index=False); os.replace(path + ".tmp", path)

def parse_month(pgn_path, month):
    rows, mv = [], []
    with open_text(pgn_path) as f:
        for tags, text in pgn_games(f):
            row = {snake(k): v for k, v in tags.items()}
            row.update(analyse_moves(text))
            rows.append(row); mv.append((tags.get("GameURL"), text))
    df = pd.DataFrame(rows)
    for c in ("white_elo", "black_elo"):
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    for c in ("white_fide_id", "black_fide_id"):
        if c in df: df[c] = df[c].astype("string")
    none = pd.Series(None, index=df.index, dtype=object)
    d = pd.to_datetime(df.get("utc_date", none), format="%Y.%m.%d", errors="coerce")
    df["game_date"] = d.fillna(pd.to_datetime(df.get("date", none), format="%Y.%m.%d", errors="coerce"))
    df["source_month"] = month
    write_parquet(df, os.path.join(INTERIM, f"games_{month}.parquet"))
    write_parquet(pd.DataFrame(mv, columns=["game_url", "movetext"]), os.path.join(INTERIM, f"moves_{month}.parquet"))
    return len(df)

def up_to_date(out, src):
    return os.path.exists(out) and os.path.getmtime(out) >= os.path.getmtime(src)

In [13]:
t0, res = time.time(), []
for month in MONTHS:
    src = os.path.join(RAW, "lichess", os.path.basename(lichess_url(month)))
    out = os.path.join(INTERIM, f"games_{month}.parquet")
    if not os.path.exists(src):
        res.append((month, "raw file missing", None)); continue
    if up_to_date(out, src) and up_to_date(os.path.join(INTERIM, f"moves_{month}.parquet"), src):
        res.append((month, "already parsed", pq.ParquetFile(out).metadata.num_rows)); continue
    res.append((month, "parsed now", parse_month(src, month)))
parsed = pd.DataFrame(res, columns=["month", "status", "games"])
display(parsed.groupby("status").agg(months=("month", "size"), games=("games", "sum")))
say(f"{parsed.games.sum():,.0f} games in {parsed.games.notna().sum()} monthly tables ({time.time() - t0:.0f} s).")

,months,games
status,,
already parsed,81,1263916


1,263,916 games in 81 monthly tables (1 s).

### 6.2 FIDE XML to `fide_<list>_YYYY-MM.parquet`

The XML is streamed with `iterparse`, so memory stays small even for lists with more than a million players. Every element is kept as text; `rating`, `games`, `k` (renamed with the list as prefix, e.g. `rapid_rating`) and `birthday` are converted to integers, and the period is taken from the file name.

In [14]:
def fide_period(name):                          # standard_jan20frl_xml.zip -> 2020-01
    m = re.search(r"(" + "|".join(MON) + r")(\d{2})", name.lower())
    return f"20{m.group(2)}-{MON.index(m.group(1)) + 1:02d}"

def parse_fide(zip_path, lst):
    rows = []
    with zipfile.ZipFile(zip_path) as z:
        member = [i.filename for i in z.infolist() if i.filename.lower().endswith(".xml")][0]
        with z.open(member) as f:
            for ev, el in ET.iterparse(f, events=("end",)):
                if el.tag == "player":
                    rows.append({c.tag: (c.text or "").strip() for c in el})
                    el.clear()
    df = pd.DataFrame(rows)
    df["fideid"] = df["fideid"].astype("string")
    df = df.rename(columns={c: f"{lst}_{c}" for c in ("rating", "games", "k") if c in df})
    for c in df.columns:
        if c == "birthday" or re.search(r"(^|_)(rating|games|k)$", c):
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    df["period"] = fide_period(os.path.basename(zip_path))
    return df

In [15]:
t0, res = time.time(), []
for month in MONTHS:
    for lst in FIDE_LISTS:
        src = os.path.join(RAW, "fide", os.path.basename(fide_url(lst, month)))
        out = os.path.join(INTERIM, f"fide_{lst}_{month}.parquet")
        if not os.path.exists(src):
            res.append((month, lst, "raw file missing", None, None)); continue
        if up_to_date(out, src):
            res.append((month, lst, "already parsed", pq.ParquetFile(out).metadata.num_rows, None)); continue
        d = parse_fide(src, lst)
        write_parquet(d, out)
        res.append((month, lst, "parsed now", len(d), int(d.fideid.duplicated().sum())))
fp = pd.DataFrame(res, columns=["month", "list", "status", "players", "duplicate_ids"])
display(fp.groupby(["list", "status"]).agg(files=("month", "size"), players_min=("players", "min"), players_max=("players", "max")))
say(f"FIDE lists parsed in {time.time() - t0:.0f} s. Duplicate FIDE ids inside a list (files parsed in this run): **{int(fp.duplicate_ids.fillna(0).sum())}**.")

,,files,players_min,players_max
list,status,,,
blitz,already parsed,81,135461,314680
rapid,already parsed,81,187401,473538
standard,already parsed,81,354588,566829


FIDE lists parsed in 2 s. Duplicate FIDE ids inside a list (files parsed in this run): **0**.

### 6.3 Combining the months

* `all_games.parquet`: every game of every month (no move text). The monthly tables do not all have the same tags (a tag that never occurs in a month is absent there); combining fills such columns with missing values and changes no value.
* `fide_ratings.parquet`: one row per (FIDE id, month) with the standard, rapid and blitz rating, games, K-factor and inactivity flag, only for players that appear in at least one game (the full lists hold more than a million players a month).
* `fide_players.parquet`: one row per FIDE id with name, federation, sex, birth year and title (latest list in which the player appears).

In [16]:
files = [os.path.join(INTERIM, f"games_{m}.parquet") for m in MONTHS if os.path.exists(os.path.join(INTERIM, f"games_{m}.parquet"))]
tables = [pq.read_table(f) for f in files]
games_tbl = pa.concat_tables(tables, promote_options="permissive").replace_schema_metadata(None)
del tables
pq.write_table(games_tbl, os.path.join(INTERIM, "all_games.parquet"))
print(f"all_games.parquet: {games_tbl.num_rows:,} games x {games_tbl.num_columns} columns from {len(files)} months")

# check: games per month against the number Lichess publishes on the database page (saved copy)
page = plain(text(saved("lichess_database_page")))
start = page.find("from the official Lichess broadcasts")
panel = page[start:page.find("Total:", start)]                    # the broadcast table only
FULL = "January February March April May June July August September October November December".split()
published = {f"{y}-{FULL.index(mo) + 1:02d}": int(n.replace(",", ""))
             for y, mo, n in re.findall(r"(\d{4}) - (\w+) [\d.]+ ?\w*B ([\d,]+)", panel) if mo in FULL}
per_month = games_tbl.group_by("source_month").aggregate([("source_month", "count")]).to_pandas()
per_month.columns = ["month", "games_parsed"]
per_month = per_month.sort_values("month").reset_index(drop=True)
per_month["games_published"] = per_month.month.map(published).astype("Int64")
per_month["difference"] = per_month.games_parsed - per_month.games_published
per_month.to_csv(os.path.join(REPORTS, "games_per_month.csv"), index=False)
off = per_month[per_month.difference.abs().fillna(0) > 5]
say(f"Games parsed per month agree with the counts published by Lichess (page saved on {os.path.basename(saved('lichess_database_page'))[-15:-5]}) "
    f"for **{int((per_month.difference.abs() <= 5).fillna(False).sum())} of {len(per_month)}** months (difference of at most 5 games; Lichess occasionally regenerates a file). "
    + ("Months that differ by more: " + ", ".join(f"{r.month} ({r.games_parsed:,} parsed, {r.games_published:,} published)" for r in off.itertuples())
       + "; re-download that file (delete it from raw/ and run section 3 again)." if len(off) else "No month differs by more."))
display(per_month.tail(6))
del games_tbl                                      # free memory before the FIDE tables are combined

all_games.parquet: 1,263,916 games x 52 columns from 81 months


Games parsed per month agree with the counts published by Lichess (page saved on 2026-10-31) for **81 of 81** months (difference of at most 5 games; Lichess occasionally regenerates a file). No month differs by more.

,month,games_parsed,games_published,difference
75,2026-04,32165,32165,0
76,2026-05,37447,37447,0
77,2026-06,36996,36996,0
78,2026-07,40038,40038,0
79,2026-08,48940,48940,0
80,2026-09,28638,28638,0


In [17]:
ids = pq.read_table(os.path.join(INTERIM, "all_games.parquet"), columns=["white_fide_id", "black_fide_id"]).to_pandas()
ids = set(ids.white_fide_id.dropna()) | set(ids.black_fide_id.dropna())
id_set = pa.array(sorted(ids), type=pa.string())
static, monthly = [], []
for p in MONTHS:
    merged = None
    for lst in FIDE_LISTS:
        f = os.path.join(INTERIM, f"fide_{lst}_{p}.parquet")
        if not os.path.exists(f): continue
        t = pq.read_table(f)
        d = t.filter(pc.is_in(t["fideid"], value_set=id_set.cast(t["fideid"].type))).to_pandas()   # players in the games only
        static.append(d[["fideid", "name", "country", "sex", "birthday", "title", "period"]])
        keep = d[["fideid"] + [c for c in (f"{lst}_rating", f"{lst}_games", f"{lst}_k", "flag") if c in d]].rename(columns={"flag": f"{lst}_flag"})
        merged = keep if merged is None else merged.merge(keep, on="fideid", how="outer")
    if merged is not None:
        merged["period"] = p; monthly.append(merged)
ratings = pd.concat(monthly, ignore_index=True)
write_parquet(ratings, os.path.join(INTERIM, "fide_ratings.parquet"))
players = pd.concat(static, ignore_index=True).sort_values("period", kind="stable").drop_duplicates("fideid", keep="last").drop(columns="period")
write_parquet(players, os.path.join(INTERIM, "fide_players.parquet"))
say(f"**{len(ids):,}** distinct FIDE ids appear in the games. `fide_ratings.parquet`: **{len(ratings):,}** rows over {ratings.period.nunique()} months; "
    f"`fide_players.parquet`: **{len(players):,}** players; ids in the games that never appear in a FIDE list: **{len(ids - set(players.fideid)):,}** "
    "(unrated players, or placeholder ids such as `0` or `-`).")
del ratings, players, static, monthly

**94,888** distinct FIDE ids appear in the games. `fide_ratings.parquet`: **4,994,547** rows over 81 months; `fide_players.parquet`: **88,503** players; ids in the games that never appear in a FIDE list: **6,385** (unrated players, or placeholder ids such as `0` or `-`).

## 7. Shape, data types and meaning of every raw column

The tables below describe the raw tables before any cleaning: one Lichess month as parsed, the combined games table, the move table, one FIDE list of each type, and the two combined FIDE tables. Shape and missing counts are read from the Parquet metadata, so the large tables are not loaded. Empty strings in the FIDE lists ("no title") are not counted as missing. The meanings are written once here and reused by the data dictionary in notebook 02 (`reports/raw_profile.csv`).

In [18]:
L, F, P = "Lichess PGN", "FIDE list", "parser (this notebook)"
RAW_MEANING = {
 "event": (L, "PGN Event tag: name of the event"), "site": (L, "PGN Site tag: place or website"),
 "date": (L, "PGN Date tag as written (may be partial, e.g. 2024.??.??)"), "round": (L, "PGN Round tag"),
 "white": (L, "White player's name as written"), "black": (L, "Black player's name as written"),
 "result": (L, "PGN Result tag (1-0, 0-1, 1/2-1/2, * and variants)"),
 "white_elo": (L, "White's rating tag (integer; non-numeric values become missing)"), "black_elo": (L, "Black's rating tag"),
 "white_title": (L, "White's FIDE title (GM, IM, ...)"), "black_title": (L, "Black's FIDE title"),
 "white_fide_id": (L, "White's FIDE id: join key to the FIDE lists"), "black_fide_id": (L, "Black's FIDE id"),
 "time_control": (L, "PGN TimeControl tag as written (e.g. 5400+30, 90+30, '90 min + 30 sec')"),
 "variant": (L, "PGN Variant tag (Standard, Chess960, ...)"), "eco": (L, "ECO opening code"), "opening": (L, "Opening name"),
 "study_name": (L, "Lichess study that holds the broadcast round"), "chapter_name": (L, "Study chapter (one board)"),
 "broadcast_name": (L, "Name of the Lichess broadcast (tournament)"), "broadcast_url": (L, "URL of the broadcast round"),
 "game_url": (L, "Lichess URL of the game; unique key of a game"),
 "utc_date": (L, "PGN UTCDate tag: date the game was played (UTC)"), "utc_time": (L, "PGN UTCTime tag: start time (UTC)"),
 "annotator": (L, "PGN Annotator tag"), "white_team": (L, "Team of White (team events)"), "black_team": (L, "Team of Black"),
 "termination": (L, "PGN Termination tag (Normal, Unterminated, Time forfeit, ...)"),
 "fen": (L, "Start position if it is not the standard one"), "set_up": (L, "PGN SetUp tag; 1 = non-standard start"),
 "board": (L, "Board number (team events)"), "chapter_url": (L, "URL of the study chapter"),
 "white_clock": (L, "PGN tag for White's clock, as written"), "black_clock": (L, "PGN tag for Black's clock, as written"),
 "game_id": (L, "Lichess game id, where given"), "white_country": (L, "Federation of White, where given"), "black_country": (L, "Federation of Black, where given"),
 "n_plies": (P, "Number of half-moves in the move text"), "has_clk": (P, "Move text contains clock readings"),
 "has_eval": (P, "Move text contains engine evaluations"),
 "first_clk_w": (P, "White's clock after its first move (seconds)"), "first_clk_b": (P, "Black's clock after its first move (seconds)"),
 "last_clk_w": (P, "White's last clock reading (seconds)"), "last_clk_b": (P, "Black's last clock reading (seconds)"),
 "n_inacc_w": (P, "White moves annotated ?! (inaccuracy)"), "n_mistake_w": (P, "White moves annotated ? (mistake)"), "n_blunder_w": (P, "White moves annotated ?? (blunder)"),
 "n_inacc_b": (P, "Black moves annotated ?!"), "n_mistake_b": (P, "Black moves annotated ?"), "n_blunder_b": (P, "Black moves annotated ??"),
 "game_date": (P, "Game date: UTCDate, else Date (missing if neither is a full date)"),
 "source_month": (P, "Month of the PGN file the game came from (YYYY-MM)"),
 "movetext": (L, "Move text with comments, as written"),
 "fideid": (F, "FIDE id (ID NUMBER)"), "name": (F, "Player name, 'Surname, Firstname'"), "country": (F, "Federation (FED)"),
 "sex": (F, "M / F"), "title": (F, "FIDE title (GM, IM, FM, CM, WGM, ...)"), "w_title": (F, "Women's title"),
 "o_title": (F, "Other titles (IA arbiter, FT trainer, ...)"), "foa_title": (F, "FIDE Online Arena title"),
 "birthday": (F, "Year of birth (B-day)"), "flag": (F, "Inactivity flag: i inactive, wi woman inactive, w woman"),
 "period": (F, "Month of the list (from the file name)"),
}
for lst in FIDE_LISTS:
    RAW_MEANING[f"{lst}_rating"] = (F, f"{lst.title()} rating in that month's list")
    RAW_MEANING[f"{lst}_games"] = (F, f"{lst.title()}-rated games played in the period")
    RAW_MEANING[f"{lst}_k"] = (F, f"{lst.title()} K-factor")
    RAW_MEANING[f"{lst}_flag"] = (F, f"Inactivity flag in the {lst} list")

def profile(path):
    pf = pq.ParquetFile(path); md_ = pf.metadata
    names = pf.schema_arrow.names
    nulls = np.zeros(len(names), dtype=np.int64)
    for i in range(md_.num_row_groups):
        for j in range(len(names)):
            st = md_.row_group(i).column(j).statistics
            nulls[j] += st.null_count if st is not None and st.null_count is not None else 0
    t = pd.DataFrame({"dtype": [str(f.type).replace("large_", "") for f in pf.schema_arrow], "missing": nulls}, index=pd.Index(names, name="column"))
    t["missing_%"] = (100 * t.missing / max(md_.num_rows, 1)).round(1)
    t["source"] = [RAW_MEANING.get(c, ("?", ""))[0] for c in names]
    t["meaning"] = [RAW_MEANING.get(c, ("?", "PGN tag (rare), kept as written"))[1] for c in names]
    return md_.num_rows, t

TABLES = [(f"games_{MONTHS[0]}.parquet", f"one Lichess month as parsed ({MONTHS[0]})"),
          ("all_games.parquet", "all games of all months"),
          (f"moves_{MONTHS[0]}.parquet", "move text of one month"),
          *[(f"fide_{l}_{MONTHS[0]}.parquet", f"FIDE {l} list of {MONTHS[0]}") for l in FIDE_LISTS],
          ("fide_ratings.parquet", "FIDE ratings, one row per player and month (players in the games only)"),
          ("fide_players.parquet", "FIDE player register, one row per player")]
prof = {}
for name, what in TABLES:
    n, t = profile(os.path.join(INTERIM, name)); prof[name] = (n, t)
undescribed = sorted({c for n, t in prof.values() for c in t.index if c not in RAW_MEANING})
pd.concat({name: t for name, (n, t) in prof.items()}, names=["table"]).to_csv(os.path.join(REPORTS, "raw_profile.csv"))
say(f"Every column has a stated meaning except: {undescribed or 'none'}.")

Every column has a stated meaning except: none.

In [19]:
n, t = prof["all_games.parquet"]
say(f"**`all_games.parquet`** (Lichess, every game parsed): **{n:,} rows × {len(t)} columns**. A single month (`games_{MONTHS[0]}.parquet`, {prof[f'games_{MONTHS[0]}.parquet'][0]:,} rows) has the same columns, "
    f"minus the tags that do not occur that month ({', '.join(sorted(set(t.index) - set(prof[f'games_{MONTHS[0]}.parquet'][1].index))) or 'none'}).")
display(t)

**`all_games.parquet`** (Lichess, every game parsed): **1,263,916 rows × 52 columns**. A single month (`games_2020-01.parquet`, 952 rows) has the same columns, minus the tags that do not occur that month (annotator, black_clock, black_country, black_team, board, chapter_url, fen, game_id, set_up, termination, white_clock, white_country, white_team).

,dtype,missing,missing_%,source,meaning
column,,,,,
event,string,0,0.0,Lichess PGN,PGN Event tag: name of the event
site,string,440910,34.9,Lichess PGN,PGN Site tag: place or website
date,string,358871,28.4,Lichess PGN,"PGN Date tag as written (may be partial, e.g. 2024.??.??)"
round,string,172705,13.7,Lichess PGN,PGN Round tag
white,string,875,0.1,Lichess PGN,White player's name as written
black,string,1019,0.1,Lichess PGN,Black player's name as written
result,string,3,0.0,Lichess PGN,"PGN Result tag (1-0, 0-1, 1/2-1/2, * and variants)"
white_elo,int64,215956,17.1,Lichess PGN,White's rating tag (integer; non-numeric values become missing)
white_title,string,829637,65.6,Lichess PGN,"White's FIDE title (GM, IM, ...)"


In [20]:
for name, what in TABLES[2:]:
    n, t = prof[name]
    say(f"`{name}` ({what}): {n:,} rows, {len(t)} columns")
    display(t)

`moves_2020-01.parquet` (move text of one month): 952 rows, 2 columns

,dtype,missing,missing_%,source,meaning
column,,,,,
game_url,string,0,0.0,Lichess PGN,Lichess URL of the game; unique key of a game
movetext,string,0,0.0,Lichess PGN,"Move text with comments, as written"


`fide_standard_2020-01.parquet` (FIDE standard list of 2020-01): 354,588 rows, 14 columns

,dtype,missing,missing_%,source,meaning
column,,,,,
fideid,string,0,0.0,FIDE list,FIDE id (ID NUMBER)
name,string,0,0.0,FIDE list,"Player name, 'Surname, Firstname'"
country,string,0,0.0,FIDE list,Federation (FED)
sex,string,0,0.0,FIDE list,M / F
title,string,0,0.0,FIDE list,"FIDE title (GM, IM, FM, CM, WGM, ...)"
w_title,string,0,0.0,FIDE list,Women's title
o_title,string,0,0.0,FIDE list,"Other titles (IA arbiter, FT trainer, ...)"
foa_title,string,0,0.0,FIDE list,FIDE Online Arena title
standard_rating,int64,0,0.0,FIDE list,Standard rating in that month's list


`fide_rapid_2020-01.parquet` (FIDE rapid list of 2020-01): 187,401 rows, 14 columns

,dtype,missing,missing_%,source,meaning
column,,,,,
fideid,string,0,0.0,FIDE list,FIDE id (ID NUMBER)
name,string,0,0.0,FIDE list,"Player name, 'Surname, Firstname'"
country,string,0,0.0,FIDE list,Federation (FED)
sex,string,0,0.0,FIDE list,M / F
title,string,0,0.0,FIDE list,"FIDE title (GM, IM, FM, CM, WGM, ...)"
w_title,string,0,0.0,FIDE list,Women's title
o_title,string,0,0.0,FIDE list,"Other titles (IA arbiter, FT trainer, ...)"
foa_title,string,0,0.0,FIDE list,FIDE Online Arena title
rapid_rating,int64,0,0.0,FIDE list,Rapid rating in that month's list


... (2 outputs left out)

`fide_ratings.parquet` (FIDE ratings, one row per player and month (players in the games only)): 4,994,547 rows, 14 columns

,dtype,missing,missing_%,source,meaning
column,,,,,
fideid,string,0,0.0,FIDE list,FIDE id (ID NUMBER)
standard_rating,int64,321404,6.4,FIDE list,Standard rating in that month's list
standard_games,int64,321404,6.4,FIDE list,Standard-rated games played in the period
standard_k,int64,321404,6.4,FIDE list,Standard K-factor
standard_flag,string,321404,6.4,FIDE list,Inactivity flag in the standard list
rapid_rating,int64,1385109,27.7,FIDE list,Rapid rating in that month's list
rapid_games,int64,1385109,27.7,FIDE list,Rapid-rated games played in the period
rapid_k,int64,1385109,27.7,FIDE list,Rapid K-factor
rapid_flag,string,1385109,27.7,FIDE list,Inactivity flag in the rapid list


`fide_players.parquet` (FIDE player register, one row per player): 88,503 rows, 6 columns

,dtype,missing,missing_%,source,meaning
column,,,,,
fideid,string,0,0.0,FIDE list,FIDE id (ID NUMBER)
name,string,0,0.0,FIDE list,"Player name, 'Surname, Firstname'"
country,string,0,0.0,FIDE list,Federation (FED)
sex,string,0,0.0,FIDE list,M / F
birthday,int64,379,0.4,FIDE list,Year of birth (B-day)
title,string,0,0.0,FIDE list,"FIDE title (GM, IM, FM, CM, WGM, ...)"


**Points to note before cleaning.** The key columns are `result`, `white_elo` / `black_elo`, `white_fide_id` / `black_fide_id`, `time_control` and the clock readings. `time_control` and the rating tags are the two with problems (missing for many games, many formats), which notebook 02 deals with. About a third of the tag columns (`annotator`, `white_team`, `board`, `white_clock`, etc.) are empty for most games and are carried along only for lineage; `fen` and `set_up` are filled only for games from a non-standard start position, which the cleaning removes. In the FIDE tables a rating is missing when the player has no rating of that type in that month.

## 8. Summary

The raw files are in `raw/` (listed with URL, date and checksum in `MANIFEST.csv`), unchanged; their tables are in `interim/`. Notebook 02_cleaning starts from `interim/all_games.parquet`, `interim/moves_*.parquet`, `interim/fide_ratings.parquet` and `interim/fide_players.parquet`.

In [21]:
def folder(path):
    files = [os.path.join(dp, f) for dp, _, fs in os.walk(path) for f in fs]
    return len(files), round(sum(os.path.getsize(f) for f in files) / 1e6, 1)
display(pd.DataFrame([(d, *folder(os.path.join(DATA_ROOT, d))) for d in ("raw", "interim", "clean")], columns=["folder", "files", "size_MB"]).set_index("folder"))

,files,size_MB
folder,,
raw,337,2620.7
interim,500,3827.3
clean,42,211.4
